# Tau-K32 condensate analysis

## Setup

In [ ]:
import glob
import itertools
import os
import pickle

import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np
import seaborn as sns
from scipy.ndimage import gaussian_filter1d

path_cond = "/home/tb/adlouet/DATA/PTM_LIBRARY/post_processed_trajectories_2025/condensates/"

modification_phosph = ['S202.T205', 'S258.262.356', 'full_phosph']
modification_acetly = ['K280', 'K274.281', 'full_ace']
unmodified = ['unmodified']
all_systems = unmodified + modification_phosph + modification_acetly

panel_labels = {'unmodified': 'Unmodified', 'S202.T205': 'Ser202/Thr205', 'S258.262.356': 'Ser258/Ser262/Ser356',
                'full_phosph': 'All Ser/Thr', 'K280': 'Lys280', 'K274.281': 'Lys274/Lys281', 'full_ace': 'All Lys'}
system_palette = ['#332288', '#117733', '#ddcc77', '#88ccee']
n_chains = 60
res_per_chain = 203
first_resid = 198

fig_dir = 'figures_condensate'
cache_dir = 'condensate_cache'  
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(cache_dir, exist_ok=True)

full_width = 7.0
sns.set_theme(style="ticks", context="paper", font="Liberation Sans", rc={
    'font.size': 10, 'axes.labelsize': 11, 'axes.titlesize': 11, 'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 10, 'axes.linewidth': 0.8, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'mathtext.default': 'regular', 'pdf.fonttype': 42, 'ps.fonttype': 42,
})

def save_figure(fig, name):
    for ext in ['pdf', 'png']:
        fig.savefig(f'{fig_dir}/{name}.{ext}', dpi=600, bbox_inches=fig.bbox_inches)
    plt.show()

def cached(name, compute):
    path = f'{cache_dir}/{name}.pck'
    if os.path.exists(path):
        with open(path, 'rb') as file:
            return pickle.load(file)
    result = compute()
    with open(path, 'wb') as file:
        pickle.dump(result, file)
    return result

def replica_dirs(y):
    if y == 'unmodified':
        base = path_cond + 'unmodified'
    elif y in modification_phosph:
        base = path_cond + f'phosph/{y}'
    else:
        base = path_cond + f'acetyl/{y}'
    return sorted(glob.glob(base + '/rep_*'))

def trajectory_files(rep_dir):
    return rep_dir + '/prod/center_traj_concatenate_1.6_us.xtc', rep_dir + '/box/condensate32_box.gro'

def chain_atoms(topology, chain):
    return topology.select(f'resid {chain * res_per_chain} to {(chain + 1) * res_per_chain - 1}')

def plot_by_modification(plot_system, xlabel, ylabel, name, xlim=None):
    fig, axes = plt.subplots(1, 2, figsize=(full_width, 3.1), sharex=True, sharey=True, constrained_layout=True)
    for ax, title, systems in zip(axes, ['Acetylated condensates', 'Phosphorylated condensates'],
                                  [modification_acetly + unmodified, modification_phosph + unmodified]):
        for y, color in zip(systems, system_palette):
            plot_system(ax, y, color)
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False,
                  columnspacing=1.0, handlelength=1.4, fontsize=9.5)
        sns.despine(ax=ax)
    axes[0].set_ylabel(ylabel)
    axes[0].set_xlim(xlim)
    save_figure(fig, name)

## Figure 3a: density profiles

In [ ]:
def mean_density(y):
    # averaged over replicas, then smoothed by one slab bin
    profiles = [np.loadtxt(rep_dir + '/prod/density_1.6_us.xvg', comments=['#', '@'], unpack=True)
                for rep_dir in replica_dirs(y)]
    return profiles[0][0], gaussian_filter1d(np.mean([density for _, density in profiles], axis=0), sigma=1)

def plot_density(ax, y, color):
    ax.plot(*mean_density(y), color=color, linewidth=1.8, label=panel_labels[y])

plot_by_modification(plot_density, 'Coordinate (nm)', r'Density (kg m$^{-3}$)', 'figure_3a_density_profiles')

## Figure 3b,c: interchain and intrachain contacts

In [ ]:
contact_cutoff_nm = 0.8
contact_stride = 200

def replica_contacts(rep_dir):
    xtc, gro = trajectory_files(rep_dir)
    traj = md.load(xtc, top=gro, stride=contact_stride)
    ca_atoms = [np.intersect1d(chain_atoms(traj.topology, i), traj.topology.select('name CA')) for i in range(n_chains)]
    intra, inter = [], []
    for i, own in enumerate(ca_atoms):
        others = np.concatenate(ca_atoms[:i] + ca_atoms[i + 1:])
        for pairs, counts in [(itertools.permutations(own, 2), intra), (itertools.product(own, others), inter)]:
            distances = md.compute_distances(traj, np.array(list(pairs)), periodic=True, opt=True)
            counts.append((distances < contact_cutoff_nm).sum(axis=1))
    return np.concatenate(intra), np.concatenate(inter)

def compute_contacts(y):
    intra, inter = zip(*map(replica_contacts, replica_dirs(y)))
    return {'intra': np.concatenate(intra), 'inter': np.concatenate(inter)}

def contact_plot(kind):
    def plot_contacts(ax, y, color):
        counts = cached(f'contacts_{y}', lambda: compute_contacts(y))[kind]
        sns.kdeplot(x=counts, color=color, linewidth=1.8, label=panel_labels[y], ax=ax)
    return plot_contacts

plot_by_modification(contact_plot('inter'), 'Interchain contacts', 'Probability density', 'figure_3b_interchain_contacts')
plot_by_modification(contact_plot('intra'), 'Intrachain contacts', 'Probability density', 'figure_3c_intrachain_contacts')

## Figure 7: per-residue coupling of intrachain and interchain contacts

In [ ]:
path_coupling = '/home/tb/adlouet/DATA/PTM_LIBRARY/analysis_notebooks/save_pickled_files/'
with open(path_coupling + 'intrachain_contacts_all_systems.pck', 'rb') as file:
    intrachain_per_residue = pickle.load(file)
with open(path_coupling + 'interchain_contacts_per_residue_all_systems.pck', 'rb') as file:
    interchain_per_residue = pickle.load(file)

def plot_contact_coupling(systems, name):
    bar_width = 0.4
    residues = np.arange(res_per_chain) + first_resid
    fig, axes = plt.subplots(len(systems), 1, figsize=(full_width, 0.45 + 1.2 * len(systems)), sharex=True, sharey=True,
                             constrained_layout=True)
    opposite = []
    for ax, y in zip(axes, systems):
        label = panel_labels[y]
        delta_intra = intrachain_per_residue['Unmodified'] - intrachain_per_residue[label]
        delta_inter = interchain_per_residue['Unmodified'] - interchain_per_residue[label]
        n_opposite = int((delta_intra * delta_inter < 0).sum())
        opposite.append(n_opposite / res_per_chain)
        print(f'{label}: {n_opposite}/{res_per_chain} residues change with opposite sign ({opposite[-1]:.1%})')
        ax.bar(residues - bar_width / 2, delta_intra, width=bar_width, color='#332288', edgecolor='none',
               label='Single chain (intrachain)')
        ax.bar(residues + bar_width / 2, delta_inter, width=bar_width, color='#117733', edgecolor='none',
               label='Condensate (interchain)')
        ax.axhline(0, color='black', linewidth=0.6)
        ax.set_title(label, loc='left', fontweight='bold', fontsize=10)
        ax.set_xlim(first_resid - 1, 401)
        sns.despine(ax=ax)
    print(f'Mean over systems: {np.mean(opposite):.1%}')
    axes[0].legend(frameon=False, loc='upper right', ncol=, bbox_to_anchor=(1, 1.35))
    axes[-1].set_xlabel('Residue number')
    fig.supylabel(r'$\Delta$ (unmodified $-$ modified)', fontsize=11)
    save_figure(fig, name)

plot_contact_coupling(modification_phosph + modification_acetly, 'figure_7_contact_coupling')

## Figure S3: per-residue secondary structure

In [ ]:
color_E = '#231526'   # Sheet (E)
color_H = '#35a1ab'   # Helix (H)
dssp_stride = 20

def compute_dssp_propensity(y):
    counts = {state: np.zeros(res_per_chain) for state in 'EHC'}
    n_samples = 0
    for rep_dir in replica_dirs(y):
        xtc, gro = trajectory_files(rep_dir)
        topology = md.load_topology(gro)
        chains = [chain_atoms(topology, i) for i in range(n_chains)]
        chain_tops = [topology.subset(atoms) for atoms in chains]
        for chunk in md.iterload(xtc, top=gro, stride=dssp_stride, chunk=100):
            for atoms, top in zip(chains, chain_tops):
                dssp = md.compute_dssp(md.Trajectory(chunk.xyz[:, atoms], top), simplified=True)
                dssp = np.where(dssp == 'NA', 'C', dssp)
                for state in counts:
                    counts[state] += (dssp == state).sum(axis=0)
                n_samples += chunk.n_frames
    return {state: counts[state] / n_samples for state in counts}

def plot_secondary_structure(systems, name):
    fig, axes = plt.subplots(len(systems), 1, figsize=(full_width, 0.45 + 1.3 * len(systems)), sharex=True, sharey=True,
                             constrained_layout=True)
    for ax, y in zip(axes, systems):
        prop = cached(f'dssp_{y}', lambda: compute_dssp_propensity(y))
        print(y, ', '.join(f'{state} {prop[state].mean():.3f} +/- {prop[state].std():.3f}' for state in 'CHE'))
        residues = np.arange(len(prop['E'])) + first_resid
        ax.bar(residues, prop['E'], width=0.8, label='Sheet (E)', color=color_E, edgecolor='none')
        ax.bar(residues, prop['H'], width=0.8, bottom=prop['E'], label='Helix (H)', color=color_H, edgecolor='none')
        ax.set_title(panel_labels[y], loc='left', fontweight='bold', fontsize=10)
        ax.set_xlim(first_resid - 1, 401)
        ax.set_ylim(0, 0.2)
        ax.set_yticks([0, 0.1, 0.2])
        sns.despine(ax=ax)
    axes[0].legend(frameon=False, loc='upper right', ncol=2, bbox_to_anchor=(1, 1.35))
    axes[-1].set_xlabel('Residue number')
    fig.supylabel('Secondary structure propensity', fontsize=11)
    save_figure(fig, name)

plot_secondary_structure(unmodified + modification_phosph, 'figure_s3a_secondary_structure_phosphorylated')
plot_secondary_structure(modification_acetly, 'figure_s3b_secondary_structure_acetylated')

## Figure S4: radius of gyration of the slab

In [ ]:
slab_time_limit_ns = 640
slab_stride = 40

def compute_slab_rg(y):
    times, rgs = [], []
    for rep_dir in replica_dirs(y):
        xtc, gro = trajectory_files(rep_dir)
        chunks = []
        for chunk in md.iterload(xtc, top=gro, stride=slab_stride, chunk=20):
            chunks.append(chunk)
            if chunk.time[-1] / 1000 >= slab_time_limit_ns:
                break
        traj = md.join(chunks)
        time_ns = traj.time / 1000
        keep = time_ns <= slab_time_limit_ns
        times.append(time_ns[keep])
        rgs.append(md.compute_rg(traj[keep], masses=None) * 10)  # nm to A
    n_frames = min(map(len, rgs))
    return times[0][:n_frames], np.array([rg[:n_frames] for rg in rgs])

def plot_slab_rg(systems, title, name):
    fig, ax = plt.subplots(figsize=(full_width, 2.8), constrained_layout=True)
    for y, color in zip(systems, system_palette):
        time_ns, rgs = cached(f'slab_rg_{y}', lambda: compute_slab_rg(y))
        mean, std = rgs.mean(axis=0), rgs.std(axis=0)
        ax.plot(time_ns, mean, color=color, linewidth=1.2, label=panel_labels[y])
        ax.fill_between(time_ns, mean - std, mean + std, color=color, alpha=0.25, linewidth=0)
    ax.set_title(title)
    ax.set_xlabel('Time (ns)')
    ax.set_ylabel(r'$R_G$ ($\AA$)')
    ax.set_xlim(0, slab_time_limit_ns)
    ax.set_ylim(100, 180)
    sns.despine(ax=ax)
    fig.legend(*ax.get_legend_handles_labels(), loc='outside lower center', ncol=4, frameon=False,
               columnspacing=1.0, handlelength=1.4, fontsize=9.5)
    save_figure(fig, name)

plot_slab_rg(modification_phosph + unmodified, 'Phosphorylated condensates', 'figure_s4a_slab_rg_phosphorylated')
plot_slab_rg(modification_acetly + unmodified, 'Acetylated condensates', 'figure_s4b_slab_rg_acetylated')